# Bank Customer Churn: Data Cleaning and Exploratory Data Analysis

This notebook loads the raw bank customer churn dataset, checks data quality, cleans column names, creates customer groups, analyzes churn patterns, and exports a cleaned dataset for SQL and Power BI analysis.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [13]:
file_path = "../data/raw/Churn_Modelling.csv"

df = pd.read_csv(file_path)

df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [14]:
df.shape

(10000, 14)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [16]:
df.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [17]:
df = df.rename(columns={
    "RowNumber": "row_number",
    "CustomerId": "customer_id",
    "Surname": "surname",
    "CreditScore": "credit_score",
    "Geography": "geography",
    "Gender": "gender",
    "Age": "age",
    "Tenure": "tenure",
    "Balance": "balance",
    "NumOfProducts": "num_of_products",
    "HasCrCard": "has_cr_card",
    "IsActiveMember": "is_active_member",
    "EstimatedSalary": "estimated_salary",
    "Exited": "exited"
})

df.head()

,row_number,customer_id,surname,credit_score,geography,gender,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [18]:
missing_values = df.isnull().sum()
missing_values

row_number          0
customer_id         0
surname             0
credit_score        0
geography           0
gender              0
age                 0
tenure              0
balance             0
num_of_products     0
has_cr_card         0
is_active_member    0
estimated_salary    0
exited              0
dtype: int64

No missing values were found in the dataset, so no missing-value imputation was required.

In [19]:
df.duplicated().sum()
df["customer_id"].duplicated().sum()

0

No duplicate customer IDs were found, so each row represents one unique customer.

In [ ]:
df["exited"].value_counts()
df["exited"].value_counts(normalize=True) * 100

In [ ]:
total_customers = len(df)
churned_customers = df["exited"].sum()
churn_rate = df["exited"].mean() * 100

print(f"Total customers: {total_customers}")
print(f"Churned customers: {churned_customers}")
print(f"Churn rate: {churn_rate:.2f}%")

In [ ]:
df["churn_status"] = df["exited"].map({
    0: "Stayed",
    1: "Churned"
})

df["credit_card_status"] = df["has_cr_card"].map({
    0: "No Credit Card",
    1: "Has Credit Card"
})

df["active_member_status"] = df["is_active_member"].map({
    0: "Inactive Member",
    1: "Active Member"
})

df.head()

In [ ]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 29, 39, 49, 59, float("inf")],
    labels=["Under 30", "30-39", "40-49", "50-59", "60+"]
)

df[["age", "age_group"]].head()

In [ ]:
df["credit_score_group"] = pd.cut(
    df["credit_score"],
    bins=[0, 579, 669, 739, 799, 900],
    labels=["Poor", "Fair", "Good", "Very Good", "Excellent"]
)

df[["credit_score", "credit_score_group"]].head()

In [ ]:
df["balance_group"] = pd.cut(
    df["balance"],
    bins=[-1, 0, 50000, 100000, 150000, 250000],
    labels=["Zero Balance", "Low Balance", "Medium Balance", "High Balance", "Very High Balance"]
)

df[["balance", "balance_group"]].head()

In [ ]:
df["salary_group"] = pd.cut(
    df["estimated_salary"],
    bins=[0, 50000, 100000, 150000, 200000],
    labels=["Low Salary", "Medium Salary", "High Salary", "Very High Salary"]
)

df[["estimated_salary", "salary_group"]].head()

In [ ]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[-1, 2, 5, 8, 10],
    labels=["0-2 Years", "3-5 Years", "6-8 Years", "9-10 Years"]
)

df[["tenure", "tenure_group"]].head()

In [ ]:
df["product_group"] = df["num_of_products"].map({
    1: "1 Product",
    2: "2 Products",
    3: "3 Products",
    4: "4 Products"
})

df[["num_of_products", "product_group"]].head()

In [ ]:
invalid_credit_scores = df[(df["credit_score"] < 300) | (df["credit_score"] > 900)]
invalid_ages = df[df["age"] <= 0]
invalid_tenure = df[(df["tenure"] < 0) | (df["tenure"] > 10)]
invalid_balance = df[df["balance"] < 0]
invalid_products = df[df["num_of_products"] <= 0]
invalid_binary_values = df[
    (~df["has_cr_card"].isin([0, 1])) |
    (~df["is_active_member"].isin([0, 1])) |
    (~df["exited"].isin([0, 1]))
]

print("Invalid credit scores:", len(invalid_credit_scores))
print("Invalid ages:", len(invalid_ages))
print("Invalid tenure values:", len(invalid_tenure))
print("Invalid balances:", len(invalid_balance))
print("Invalid product counts:", len(invalid_products))
print("Invalid binary values:", len(invalid_binary_values))

No invalid values were found in the main numeric and binary fields. Credit scores, ages, tenure values, balances, product counts, and churn labels were within expected ranges.

In [ ]:
df[[
    "credit_score",
    "age",
    "tenure",
    "balance",
    "num_of_products",
    "estimated_salary",
    "exited"
]].describe()

In [ ]:
churn_by_geography = (
    df.groupby("geography")
    .agg(
        total_customers=("customer_id", "count"),
        churned_customers=("exited", "sum"),
        churn_rate=("exited", "mean")
    )
    .reset_index()
)

churn_by_geography["churn_rate"] = churn_by_geography["churn_rate"] * 100

churn_by_geography.sort_values("churn_rate", ascending=False)

In [ ]:
churn_by_gender = (
    df.groupby("gender")
    .agg(
        total_customers=("customer_id", "count"),
        churned_customers=("exited", "sum"),
        churn_rate=("exited", "mean")
    )
    .reset_index()
)

churn_by_gender["churn_rate"] = churn_by_gender["churn_rate"] * 100

churn_by_gender.sort_values("churn_rate", ascending=False)

In [ ]:
churn_by_age_group = (
    df.groupby("age_group", observed=True)
    .agg(
        total_customers=("customer_id", "count"),
        churned_customers=("exited", "sum"),
        churn_rate=("exited", "mean")
    )
    .reset_index()
)

churn_by_age_group["churn_rate"] = churn_by_age_group["churn_rate"] * 100

churn_by_age_group

In [ ]:
churn_by_active_status = (
    df.groupby("active_member_status")
    .agg(
        total_customers=("customer_id", "count"),
        churned_customers=("exited", "sum"),
        churn_rate=("exited", "mean")
    )
    .reset_index()
)

churn_by_active_status["churn_rate"] = churn_by_active_status["churn_rate"] * 100

churn_by_active_status.sort_values("churn_rate", ascending=False)

In [ ]:
churn_by_products = (
    df.groupby("product_group")
    .agg(
        total_customers=("customer_id", "count"),
        churned_customers=("exited", "sum"),
        churn_rate=("exited", "mean")
    )
    .reset_index()
)

churn_by_products["churn_rate"] = churn_by_products["churn_rate"] * 100

churn_by_products.sort_values("churn_rate", ascending=False)

In [ ]:
churn_by_balance_group = (
    df.groupby("balance_group", observed=True)
    .agg(
        total_customers=("customer_id", "count"),
        churned_customers=("exited", "sum"),
        churn_rate=("exited", "mean")
    )
    .reset_index()
)

churn_by_balance_group["churn_rate"] = churn_by_balance_group["churn_rate"] * 100

churn_by_balance_group

In [ ]:
churn_by_credit_score_group = (
    df.groupby("credit_score_group", observed=True)
    .agg(
        total_customers=("customer_id", "count"),
        churned_customers=("exited", "sum"),
        churn_rate=("exited", "mean")
    )
    .reset_index()
)

churn_by_credit_score_group["churn_rate"] = churn_by_credit_score_group["churn_rate"] * 100

churn_by_credit_score_group

In [ ]:
churn_counts = df["churn_status"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(churn_counts.index, churn_counts.values)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn Status")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
geo_plot = churn_by_geography.sort_values("churn_rate", ascending=False)

plt.figure(figsize=(7, 4))
plt.bar(geo_plot["geography"], geo_plot["churn_rate"])
plt.title("Churn Rate by Geography")
plt.xlabel("Geography")
plt.ylabel("Churn Rate (%)")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(churn_by_age_group["age_group"].astype(str), churn_by_age_group["churn_rate"])
plt.title("Churn Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Churn Rate (%)")
plt.show()

In [ ]:
active_plot = churn_by_active_status.sort_values("churn_rate", ascending=False)

plt.figure(figsize=(7, 4))
plt.bar(active_plot["active_member_status"], active_plot["churn_rate"])
plt.title("Churn Rate by Active Membership Status")
plt.xlabel("Membership Status")
plt.ylabel("Churn Rate (%)")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(churn_by_products["product_group"], churn_by_products["churn_rate"])
plt.title("Churn Rate by Number of Products")
plt.xlabel("Product Group")
plt.ylabel("Churn Rate (%)")
plt.show()

In [ ]:
cleaned_file_path = "../data/cleaned/bank_churn_cleaned.csv"

df.to_csv(cleaned_file_path, index=False)

print(f"Cleaned dataset saved to: {cleaned_file_path}")

In [ ]:
churn_by_geography.to_csv("../data/cleaned/churn_by_geography.csv", index=False)
churn_by_gender.to_csv("../data/cleaned/churn_by_gender.csv", index=False)
churn_by_age_group.to_csv("../data/cleaned/churn_by_age_group.csv", index=False)
churn_by_active_status.to_csv("../data/cleaned/churn_by_active_status.csv", index=False)
churn_by_products.to_csv("../data/cleaned/churn_by_products.csv", index=False)
churn_by_balance_group.to_csv("../data/cleaned/churn_by_balance_group.csv", index=False)
churn_by_credit_score_group.to_csv("../data/cleaned/churn_by_credit_score_group.csv", index=False)

## Initial Findings

- The dataset contains 10,000 customer records.
- The overall churn rate is approximately 20.37%.
- No missing values were found.
- No duplicate customer IDs were found.
- Churn appears to vary by geography, age group, activity status, product usage, and balance group.
- Inactive customers appear to have a higher churn rate than active customers.
- These patterns will be explored more deeply using SQL and Power BI in the next phases.